# 07_attention: Query-Key-Value Matrix Calculations from Scratch

This notebook programmatically computes Self-Attention Query-Key-Value dot-product matrix transformations, demonstrating the variance scaling effect of dividing by $\sqrt{d_k}$.


In [1]:
import torch
import torch.nn.functional as F

# Sequence length L=3, dimension d_k=64
torch.manual_seed(42)
L, d_k = 3, 64

# Simulating Query and Key inputs
Q = torch.randn(L, d_k)
K = torch.randn(L, d_k)
V = torch.randn(L, d_k)

# 1. Unscaled Attention
scores_unscaled = torch.matmul(Q, K.T)
weights_unscaled = F.softmax(scores_unscaled, dim=-1)

# 2. Scaled Attention (dividing by sqrt(d_k))
scaling_factor = d_k ** 0.5
scores_scaled = scores_unscaled / scaling_factor
weights_scaled = F.softmax(scores_scaled, dim=-1)

print("=== Raw Similarity Scores ===")
print(scores_unscaled)

print("\n=== Unscaled Softmax Attention Weights (Variance is high, scores saturate) ===")
print(weights_unscaled)
print("Weights variance:", torch.var(weights_unscaled).item())

print("\n=== Scaled Softmax Attention Weights (Normalized variance, sensitive gradients) ===")
print(weights_scaled)
print("Weights variance:", torch.var(weights_scaled).item())


=== Raw Similarity Scores ===
tensor([[-2.4180, -2.1061,  9.6773],
        [ 8.7938, -2.4409, -1.7275],
        [ 2.2382,  2.7408, 11.6922]])

=== Unscaled Softmax Attention Weights (Variance is high, scores saturate) ===
tensor([[5.5854e-06, 7.6299e-06, 9.9999e-01],
        [9.9996e-01, 1.3208e-05, 2.6954e-05],
        [7.8362e-05, 1.2953e-04, 9.9979e-01]])
Weights variance: 0.24993468821048737

=== Scaled Softmax Attention Weights (Normalized variance, sensitive gradients) ===
tensor([[0.1521, 0.1581, 0.6898],
        [0.6605, 0.1622, 0.1773],
        [0.1878, 0.2000, 0.6122]])
Weights variance: 0.058504700660705566


### Output Explanation
- Unscaled dot products yield larger values, pushing Softmax inputs into saturating regions where the output weights approach $0$ or $1$, causing vanishing gradients.
- Dividing by $\sqrt{d_k}$ restores unit variance, keeping the weights in a range where gradients can flow during training.
